# F02-P4 Pathway

**Pathway Recommendation, components 4.x.**

Reads the pre-computed pathway raster and reports which NBS pathways the project area qualifies
for, and how much of it each one covers.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** 4.1 Pathway Distribution is written, on the canonical_v3 raster layout.

## Where the pathway raster comes from

This notebook does not assign pathways. The assignment happens upstream, in
`nbs_trajectory_pathway_v3.js`, and arrives here as a finished three band raster. The decision
matrix, the three rules behind it and the savanna guardrail are documented in the canonical_v3
pathway logic, not here. This notebook only summarises the result over an AOI.

**Built for canonical_v3, locked 2026-07-24.** The band layout changed from v2 and the two are
not compatible. In v2 band 2 was a secondary pathway and band 3 was the ecosystem. In v3 there
is no secondary pathway: band 2 is the ecosystem and band 3 is the 17 class category index.
Reading a v2 raster with this notebook would silently swap the ecosystem and category bands, so
the raster and the notebook version must match.

## Handoff

Reads nothing from earlier stages. Writes `outputs/<aoi_id>__F02-P4-pathway.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass

import geopandas as gpd
import numpy as np

from config import *
from common import *

In [ ]:
AOI_PATH = r"<SET: path to the project AOI polygon>"
aoi_id = "<SET: short id for this run, must match the F02-P2 notebooks>"

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

---
## 4.1 Pathway Distribution

Reports the area in hectares and the share of the AOI for each pathway.

**Data.** `pathway.tif`, one raster with three bands, canonical_v3 layout.

| Band | Name | Contents |
|---|---|---|
| 1 | `pathway` | primary pathway, exactly one value per pixel |
| 2 | `ecosystem` | reference ecosystem, passed through for activity selection |
| 3 | `cat_code` | 1 to 17 canonical_v3 category index, passed through for activities |

Band 1 codes: 0 no data, 1 Protect, 2 Manage, 3 Restore, 4 Carbon ineligible.

**Code 4 is not a pathway, and it is not an error.** It sits in the same band because every
pixel needs exactly one primary value. "Carbon ineligible" means the site cannot generate carbon
credits, though non carbon options may still exist. It gathers several v3 categories:
established plantation, savanna that lost its forest cover, stable natural savanna, and
settlement. An AOI that is 60% oil palm will report 60% Carbon ineligible, and that is the
screening result, not a failure of the layer. It is listed in the table but excluded from the
eligible area headline.

**The v2 code 5 "Not eligible for NBS" is gone.** In v3 settlement folds into code 4, so the
primary band runs 0 to 4 only. A raster that still carries a code 5 is a v2 product and does not
belong here.

**Decisions locked.**

- Primary band only. The team chose a single mutually exclusive breakdown, so the shares sum to
  100 and each hectare is counted once. v3 has no secondary band to report.
- Denominator is the **total AOI area**, not the classified area. A user asking for "the
  proportion of each pathway in my AOI" means a share of their site. An Unclassified row absorbs
  code 0 and nodata so the table still sums to 100. Without this an AOI that is 90% outside the
  layer would report "50% Protect" of the 10% that happens to be covered, which reads as a
  finding about the site and is not one.
- Resampling is `nearest`. The bands are categorical.
- Eligible NBS area is the sum of codes 1 to 3, reported as the headline. Code 4 is excluded
  from it by definition.
- The component flags an AOI where the unclassified share passes
  `PATHWAY_UNCLASSIFIED_WARN_PCT`, because every percentage below that point describes a
  shrinking part of the site.

**Bands 2 and 3 are read but not tabulated.** The ecosystem and the category index travel into
`values` as the sets of codes present, for the activity generator downstream (F02-P5, keyed on
`(cat_code, ecosystem)`). The ecosystem band has four classes plus none, not the three used by
1.1: it carries savanna, which is what the savanna guardrail needs, since RESTORE on a savanna
reference means seeding native grasses rather than planting trees. The two ecosystem layers are
not interchangeable and must not be cross-read.

**Example render.**

> Of the 1,240 ha project area, 1,190 ha (96%) qualifies for a Nature-Based Solutions pathway.
> Manage covers the largest share at 620 ha (50%), followed by Restore at 370 ha (30%) and
> Protect at 200 ha (16%). A further 50 ha (4%) is carbon ineligible.

**Narrative not yet specified.** Placeholder wording. Replace it once the team settles the
phrasing.

**Downstream use.** The pathway split drives Level 3 activity selection, which reads this raster
rather than the trajectory raster, and it sets the scope of Benefit Quantification in F02-P5:
avoided loss applies to the Protect area, improved management to the Manage area, and removals to
the Restore area. F02-P5 joins the `cat_code` and `ecosystem` bands to the canonical_v3_activities
table, which is why both bands are carried through here.

In [ ]:
@dataclass(frozen=True)
class PathwayShare:
    """One row of the pathway breakdown."""

    code: int
    label: str
    area_ha: float
    pct: float          # share of the TOTAL AOI area, not of the classified area
    is_pathway: bool    # False for Carbon ineligible and Unclassified


UNCLASSIFIED_LABEL = "Unclassified"


def _codes_present(aoi: AOI, band: int, labels: dict[int, str]) -> dict:
    """Read one pass-through band and return which codes occur, with labels and areas.

    Used for the ecosystem and cat_code bands. They are not tabulated into the headline, but the
    activity generator needs to know which categories the AOI contains, so the set of present
    codes and their areas travel in `values`.
    """
    raster = load_raster_clipped(PATHWAY_RASTER, aoi, resampling="nearest", band=band)
    codes, counts = np.unique(raster.values.compressed(), return_counts=True)
    area_by_code = {
        int(c): int(n) * raster.pixel_area_ha
        for c, n in zip(codes.tolist(), counts.tolist())
        if int(c) != 0  # 0 is mask / none, not a category
    }
    return {
        "codes": sorted(area_by_code),
        "labels": [labels.get(c, f"Unknown code {c}") for c in sorted(area_by_code)],
        "area_ha": {labels.get(c, f"Unknown code {c}"): area_by_code[c]
                    for c in sorted(area_by_code)},
    }


def analyze_pathway_distribution(aoi: AOI) -> ComponentResult:
    """Component 4.1. Area and share of the AOI per primary pathway, canonical_v3."""
    primary = load_raster_clipped(
        PATHWAY_RASTER, aoi, resampling="nearest", band=PATHWAY_BAND
    )

    if primary.valid_area_ha <= 0:
        return not_applicable(
            "4.1 Pathway Distribution",
            "The pathway layer does not cover this project area, so no pathway can be "
            "recommended.",
        )

    # Denominator is the whole site. Code 0 and nodata are absorbed by an Unclassified row, so
    # the table sums to 100 of the AOI rather than of the covered part.
    rows: list[PathwayShare] = []
    classified_ha = 0.0
    for code, label in PATHWAY_CODES.items():
        if code == 0:
            continue  # 0 is folded into Unclassified below, together with nodata
        area_ha = int((primary.values == code).sum()) * primary.pixel_area_ha
        classified_ha += area_ha
        rows.append(
            PathwayShare(
                code=code,
                label=label,
                area_ha=area_ha,
                pct=safe_pct(area_ha, aoi.area_ha),
                is_pathway=code in PATHWAY_ELIGIBLE_CODES,
            )
        )

    unclassified_ha = max(0.0, aoi.area_ha - classified_ha)
    rows.append(
        PathwayShare(
            code=0,
            label=UNCLASSIFIED_LABEL,
            area_ha=unclassified_ha,
            pct=safe_pct(unclassified_ha, aoi.area_ha),
            is_pathway=False,
        )
    )

    eligible_ha = sum(r.area_ha for r in rows if r.is_pathway)
    eligible_pct = safe_pct(eligible_ha, aoi.area_ha)

    flags: list[str] = []
    unclassified_pct = safe_pct(unclassified_ha, aoi.area_ha)
    if unclassified_pct > PATHWAY_UNCLASSIFIED_WARN_PCT:
        flags.append(
            f"4.1: {unclassified_pct:.0f}% of the AOI carries no pathway value. Every share "
            "below describes only the remainder of the site."
        )

    # Bands 2 and 3 pass through untabulated, for the activity generator in F02-P5.
    ecosystem = _codes_present(aoi, PATHWAY_ECOSYSTEM_BAND, PATHWAY_ECOSYSTEM_CODES)
    catcode = _codes_present(aoi, PATHWAY_CATCODE_BAND, PATHWAY_CATCODE_LABELS)

    pathway_rows = sort_by_area([r for r in rows if r.is_pathway and r.area_ha > 0])
    other_rows = [r for r in rows if not r.is_pathway and r.area_ha > 0]

    # Placeholder wording, see the note above.
    if pathway_rows:
        listed = oxford_join(
            f"{r.label} at {fmt_ha(r.area_ha)} ({fmt_pct(r.pct)})" for r in pathway_rows
        )
        head = (
            f"Of the {fmt_ha(aoi.area_ha)} project area, {fmt_ha(eligible_ha)} "
            f"({fmt_pct(eligible_pct)}) qualifies for a Nature-Based Solutions pathway. "
            f"The split is {listed}."
        )
    else:
        head = (
            f"None of the {fmt_ha(aoi.area_ha)} project area qualifies for a Nature-Based "
            "Solutions pathway."
        )

    tail = ""
    if other_rows:
        tail = "A further " + oxford_join(
            f"{fmt_ha(r.area_ha)} ({fmt_pct(r.pct)}) is {r.label.lower()}" for r in other_rows
        ) + "."

    return ComponentResult(
        component="4.1 Pathway Distribution",
        applicable=True,
        narrative=sentences(head, tail),
        tables={"pathway_distribution": rows},  # every code plus Unclassified, sums to 100
        values={
            "chart_series": "pathway_distribution",
            "chart_unit": "%",
            "chart_axis_label": "Share of project area (%)",
            "eligible_ha": eligible_ha,
            "eligible_pct": eligible_pct,
            "pathway_ha": {r.label: r.area_ha for r in rows if r.is_pathway},
            "dominant_pathway": pathway_rows[0].label if pathway_rows else None,
            "unclassified_pct": unclassified_pct,
            # Band 2 ecosystem and band 3 cat_code, for the F02-P5 activity join on
            # (cat_code, ecosystem). Four class ecosystem, NOT the three class layer of 1.1.
            "reference_ecosystem_codes": ecosystem["codes"],
            "reference_ecosystem_labels": ecosystem["labels"],
            "cat_code_codes": catcode["codes"],
            "cat_code_labels": catcode["labels"],
        },
        flags=flags,
    )

---
## Run and save

In [ ]:
results: dict[str, ComponentResult] = {}

results["4.1"] = analyze_pathway_distribution(aoi)

for key, r in results.items():
    print(f"[{key}] {r.component}{'' if r.applicable else '  (not applicable)'}")
    print(f"      {r.narrative}")
    for f in r.flags:
        print(f"      FLAG: {f}")

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_PATHWAY)
print(f"Saved {path}")